# Bird XAI — White-fronted Geese 전처리
## North Sea population tracks 2014-2017

## 처리 방식
Drive에 업로드된 ERA5 파일을 읽어 전처리 → 결과 CSV를 Drive에 저장
연도별로 반복 (Cell 4의 YEAR만 바꾸고 Cell 5~10 재실행)

## Drive 파일 구조
```
MyDrive/data/
  North_Sea_population_tracks_...csv     ← GPS 원본
  era5_geese/
    era5_geese_2014_09.nc  (또는 _w1~w4)
    era5_geese_2014_10.nc
    era5_geese_2014_11.nc
    ...  (4개 연도 × 3개월)
  lstm_input_geese/
    preprocessed_geese_2014.csv          ← 처리 결과
    preprocessed_geese_2015.csv
    ...
    preprocessed_geese_full.csv          ← 최종 합본
```

## 연도별 개체 수
| 연도 | 개체 수 |
|------|--------|
| 2014 | 13마리 |
| 2015 | 10마리 |
| 2016 | 20마리 |
| 2017 | 22마리 |

## 최종 CSV 컬럼 (44개)
- **메타 (4)**: bird, timestamp, is_interpolated_gps, is_interpolated_era5
- **y 타깃 (3)**: lat, lon, height_raw
- **GPS 행동 (3)**: ground_speed, heading, is_moving
- **ERA5 원본 (24)**: u/v/w/t/z/q/r/cc × 1000/925/850hPa
- **파생 변수 (10)**: ws/wspeed/wdir × 3레벨 + lapse_rate

## 실행 순서
1. Cell 1~3: 최초 1회 실행
2. Cell 4: YEAR 설정 (2014 → 2015 → 2016 → 2017)
3. Cell 5~10: 연도별 반복 실행
4. 모든 연도 완료 후 Cell 11: 합치기

In [1]:
# ── Cell 1: 설치 및 세션 유지 ─────────────────────────────────────────
!pip install astral xarray netCDF4 scikit-learn -q
print('설치 완료!')

from IPython.display import Javascript
display(Javascript('''
function KeepAlive() {
  document.querySelector('#top-toolbar').click();
  console.log('Session alive:', new Date());
  setTimeout(KeepAlive, 60000);
}
KeepAlive();
console.log('세션 유지 시작 (60초마다)');
'''))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.1 MB/s eta 0:00:00
설치 완료!


<IPython.core.display.Javascript object>

In [8]:
# ── Cell 2: Google Drive 연결 및 경로 설정 ────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# ▼ 경로 설정 (필요시 수정)
GPS_CSV  = '/content/drive/MyDrive/data/North Sea population tracks of greater white-fronted geese 2014-2017 (data from Klzsch et al. 2019).csv'
ERA5_DIR = Path('/content/drive/MyDrive/data/era5_geese')   # ERA5 파일 폴더
OUT_DIR  = Path('/content/drive/MyDrive/data/lstm_input_geese')  # 결과 저장

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'GPS    : {GPS_CSV}')
print(f'ERA5   : {ERA5_DIR}')
print(f'출력   : {OUT_DIR}')

# ERA5 파일 확인
era5_files = sorted(ERA5_DIR.glob('*.nc'))
print(f'\nERA5 파일 수: {len(era5_files)}개')
for f in era5_files:
    print(f'  {f.name}  ({f.stat().st_size/1024**2:.0f} MB)')

# 기존 완료 파일 확인
done = sorted(OUT_DIR.glob('preprocessed_geese_20??.csv'))
print(f'\n기존 완료: {len(done)}개')
for f in done:
    print(f'  {f.name}  ({f.stat().st_size/1024**2:.1f} MB)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPS    : /content/drive/MyDrive/data/North Sea population tracks of greater white-fronted geese 2014-2017 (data from Klzsch et al. 2019).csv
ERA5   : /content/drive/MyDrive/data/era5_geese
출력   : /content/drive/MyDrive/data/lstm_input_geese

ERA5 파일 수: 12개
  era5_geese_2014_09_w1.nc  (400 MB)
  era5_geese_2014_09_w2.nc  (403 MB)
  era5_geese_2014_09_w3.nc  (401 MB)
  era5_geese_2014_09_w4.nc  (516 MB)
  era5_geese_2014_10_w1.nc  (404 MB)
  era5_geese_2014_10_w2.nc  (399 MB)
  era5_geese_2014_10_w3.nc  (404 MB)
  era5_geese_2014_10_w4.nc  (580 MB)
  era5_geese_2014_11_w1.nc  (410 MB)
  era5_geese_2014_11_w2.nc  (408 MB)
  era5_geese_2014_11_w3.nc  (409 MB)
  era5_geese_2014_11_w4.nc  (528 MB)

기존 완료: 0개


In [9]:
# ── Cell 3: 라이브러리 및 함수 정의 ──────────────────────────────────
import gc
import pandas as pd
import numpy as np
import xarray as xr


def circular_mean(angles_deg):
    """heading circular mean (각도 평균)"""
    a = angles_deg.dropna()
    if len(a) == 0:
        return np.nan
    r = np.radians(a)
    return float(np.degrees(np.arctan2(np.sin(r).mean(), np.cos(r).mean())) % 360)


def bearing(lat1, lon1, lat2, lon2):
    """두 좌표 간 방위각 계산 (0~360°)"""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1)*np.sin(lat2) - np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


def wind_support(u, v, heading_deg):
    """순풍지원: u·sin(heading) + v·cos(heading). 양수=순풍, 음수=역풍"""
    r = np.radians(heading_deg.fillna(180))
    return u * np.sin(r) + v * np.cos(r)


def wind_dir(u, v):
    """바람 방향 (0~360°)"""
    return (np.degrees(np.arctan2(u, v)) + 360) % 360


def load_era5_month(year, month):
    """ERA5 월별 로드. 주 단위 파일이면 자동 병합."""
    p = ERA5_DIR / f'era5_geese_{year}_{month:02d}.nc'
    if p.exists():
        return xr.open_dataset(p, engine='netcdf4', chunks={'valid_time': 24})
    # 주 단위 파일 병합
    datasets = []
    for i in range(1, 5):
        wp = ERA5_DIR / f'era5_geese_{year}_{month:02d}_w{i}.nc'
        if wp.exists():
            datasets.append(xr.open_dataset(wp, engine='netcdf4',
                                            chunks={'valid_time': 24}))
    if not datasets:
        print(f'  [ERROR] ERA5 {year}-{month:02d} 파일 없음')
        return None
    return xr.concat(datasets, dim='valid_time').sortby('valid_time')


print('함수 정의 완료!')

함수 정의 완료!


In [17]:
# ── Cell 4: 처리할 연도 설정 ─────────────────────────────────────────
# ▼ 여기만 바꾸고 Cell 5~10 실행
YEAR = 2015  # 2014 → 2015 → 2016 → 2017

YEAR_BIRDS = {
    2014: ['701','707','709','716','720','742','749','750','768',
           'CS007027_3098','Frank_3084','Gustaf_3104','Ronja_3271_KY5'],
    2015: ['711','712','738','766','Hannah_3988_LK2','HeBerend_4000_NK3',
           'Jouri_3992_BG1','NilsHolgerson_11','Olga_13','Ontsje_12'],
    2016: ['GWFG_2015_408','GWFG_2015_409','GWFG_2015_411','GWFG_2015_412',
           'GWFG_2015_414','GWFG_2015_419','GWFG_2015_420','GWFG_2015_421',
           'GWFG_2015_423','GWFG_2015_425','GWFG_2015_427','GWFG_2015_428(A)',
           'GWFG_2015_432','GWFG_2015_442','GWFG_2015_443','GWFG_2015_447',
           'GWFG_2015_448','GWFG_2015_449','GWFG_2015_450','HeBerend_4000_NK3'],
    2017: ['52_EvertII_M','58_WouterII_M','71_Jan_M','82_Chris_M',
           'GWFG_2015_410','GWFG_2015_413(B)','GWFG_2015_416',
           'GWFG_2015_429','GWFG_2015_430','GWFG_2015_431',
           'GWFG_2015_438','GWFG_2015_440','GWFG_2015_445',
           'KOL_02_M','KOL_23_M','KOL_28_M','KOL_31_F',
           'KOL_35_F','KOL_36_F','KOL_37_F','KOL_41_F','KOL_42_F'],
}

out_csv = OUT_DIR / f'preprocessed_geese_{YEAR}.csv'

if out_csv.exists():
    mb = out_csv.stat().st_size / 1024**2
    print(f'[SKIP] {YEAR}년 이미 완료: {out_csv.name} ({mb:.1f} MB)')
    print(f'다음 연도로 변경하세요: YEAR = {YEAR+1 if YEAR < 2017 else "완료"}')
else:
    print(f'처리 대상: {YEAR}년 ({len(YEAR_BIRDS[YEAR])}마리)')
    print(f'개체: {YEAR_BIRDS[YEAR]}')
    print(f'저장 경로: {out_csv}')

처리 대상: 2015년 (10마리)
개체: ['711', '712', '738', '766', 'Hannah_3988_LK2', 'HeBerend_4000_NK3', 'Jouri_3992_BG1', 'NilsHolgerson_11', 'Olga_13', 'Ontsje_12']
저장 경로: /content/drive/MyDrive/data/lstm_input_geese/preprocessed_geese_2015.csv


In [18]:
# ── Cell 5: GPS 로드 및 1시간 단위 집계 ──────────────────────────────
print(f'GPS 로드 및 1시간 집계 중... ({YEAR}년)')

raw = pd.read_csv(GPS_CSV, low_memory=False)
raw['timestamp']        = pd.to_datetime(raw['timestamp'])
raw['ground-speed']     = pd.to_numeric(raw['ground-speed'],     errors='coerce')
raw['heading']          = pd.to_numeric(raw['heading'],          errors='coerce')
raw['height-above-msl'] = pd.to_numeric(raw['height-above-msl'], errors='coerce')

# 해당 연도 + 9~11월 필터
gps = raw[
    (raw['individual-local-identifier'].isin(YEAR_BIRDS[YEAR])) &
    (raw['timestamp'].dt.year == YEAR) &
    (raw['timestamp'].dt.month.isin([9, 10, 11]))
].copy().rename(columns={
    'individual-local-identifier': 'bird',
    'location-lat':    'lat',
    'location-long':   'lon',
    'height-above-msl':'height_raw',
    'ground-speed':    'ground_speed',
})[['bird','timestamp','lat','lon','height_raw','ground_speed','heading']]

# 이상치 처리
gps['height_raw'] = gps['height_raw'].clip(lower=0)
gps.loc[(gps['ground_speed'] <= 1) & (gps['heading'] == 0), 'heading'] = np.nan

# 1시간 단위 집계
gps['timestamp_1h'] = gps['timestamp'].dt.floor('1h')
gps_1h_list = []
for bird, g in gps.groupby('bird'):
    g = g.sort_values('timestamp')
    agg = g.groupby('timestamp_1h').agg(
        lat          = ('lat',          'mean'),
        lon          = ('lon',          'mean'),
        height_raw   = ('height_raw',   'mean'),
        ground_speed = ('ground_speed', 'mean'),
        heading      = ('heading', lambda x: circular_mean(x)),
    ).reset_index().rename(columns={'timestamp_1h': 'timestamp'})
    agg['bird'] = bird
    agg['is_interpolated_gps'] = 0
    gps_1h_list.append(agg)

gps_1h = pd.concat(gps_1h_list, ignore_index=True)
gps_1h = gps_1h.sort_values(['bird','timestamp']).reset_index(drop=True)
print(f'1시간 집계 완료: {len(gps_1h):,}건 ({gps_1h["bird"].nunique()}마리)')

GPS 로드 및 1시간 집계 중... (2015년)
1시간 집계 완료: 6,037건 (10마리)


In [19]:
# ── Cell 6: 1시간 리샘플링 + 혼합 보간 ───────────────────────────────
# lat/lon/height_raw → spline(3차)
# ground_speed/heading → linear
# 24시간 초과 gap → NaN 유지

MAX_GAP_H = 24
print('리샘플링 + 보간 중...')
resampled_list = []

for bird, g in gps_1h.groupby('bird'):
    g = g.set_index('timestamp').sort_index()
    full_range = pd.date_range(start=g.index.min(),
                               end=g.index.max(), freq='1h')
    g = g.reindex(full_range)
    g['is_interpolated_gps'] = g['lat'].isna().astype(int)

    # 24h 초과 gap 마스크
    interp_ok = pd.Series(True, index=g.index)
    measured_idx = g.index[g['lat'].notna()]
    for i in range(len(measured_idx)-1):
        gap_h = (measured_idx[i+1]-measured_idx[i]).total_seconds()/3600
        if gap_h > MAX_GAP_H:
            mask = (g.index > measured_idx[i]) & (g.index < measured_idx[i+1])
            interp_ok[mask] = False

    # spline 보간: lat, lon, height_raw
    for col in ['lat','lon','height_raw']:
        temp = g[col].copy()
        temp[~interp_ok] = np.nan
        try:
            g[col] = temp.interpolate(method='spline', order=3,
                                      limit_direction='both')
        except Exception:
            g[col] = temp.interpolate(method='linear', limit_direction='both')
        g.loc[~interp_ok, col] = np.nan

    # linear 보간: ground_speed, heading
    for col in ['ground_speed','heading']:
        temp = g[col].copy()
        temp[~interp_ok] = np.nan
        g[col] = temp.interpolate(method='linear', limit_direction='both')
        g.loc[~interp_ok, col] = np.nan

    # 보간 후 음수 재클리핑
    g['height_raw'] = g['height_raw'].clip(lower=0)

    # heading NaN → bearing으로 대체
    g_reset = g.reset_index().rename(columns={'index':'timestamp'})
    for i in range(1, len(g_reset)):
        if pd.isna(g_reset.loc[i,'heading']) and pd.notna(g_reset.loc[i,'lat']):
            if pd.notna(g_reset.loc[i-1,'lat']):
                g_reset.loc[i,'heading'] = bearing(
                    g_reset.loc[i-1,'lat'], g_reset.loc[i-1,'lon'],
                    g_reset.loc[i,'lat'],   g_reset.loc[i,'lon']
                )

    g_reset['bird'] = bird
    resampled_list.append(g_reset)
    print(f'  {bird}: {len(g_reset)}건 (보간 {g_reset["is_interpolated_gps"].sum()}건)')

gps_resampled = pd.concat(resampled_list, ignore_index=True)
print(f'\n리샘플링 완료: 총 {len(gps_resampled):,}건')

리샘플링 + 보간 중...
  711: 1815건 (보간 900건)
  712: 1817건 (보간 1544건)
  738: 1787건 (보간 1580건)
  766: 1823건 (보간 802건)
  Hannah_3988_LK2: 5건 (보간 3건)
  HeBerend_4000_NK3: 1815건 (보간 572건)
  Jouri_3992_BG1: 1503건 (보간 765건)
  NilsHolgerson_11: 1775건 (보간 1393건)
  Olga_13: 1823건 (보간 1117건)
  Ontsje_12: 1811건 (보간 1261건)

리샘플링 완료: 총 15,974건


In [ ]:
# ── Cell 7: ERA5 매칭 ─────────────────────────────────────────────────
# Drive에서 ERA5 파일 읽어 GPS 포인트에 매칭
# 연도 처리 중 중단 시 월별로 중간 저장하여 이어받기 가능

print(f'ERA5 매칭 시작... ({YEAR}년)')
era5_records = []

for month in [9, 10, 11]:
    # 월별 중간 저장 파일 확인 (이어받기용)
    cache_path = OUT_DIR / f'era5_matched_{YEAR}_{month:02d}.csv'
    if cache_path.exists():
        print(f'  [{YEAR}-{month:02d}] 캐시 파일 존재 → SKIP')
        cached = pd.read_csv(cache_path)
        cached['timestamp'] = pd.to_datetime(cached['timestamp'])
        era5_records.append(cached)
        continue

    print(f'  [{YEAR}-{month:02d}] ERA5 로드 중...')
    ds = load_era5_month(YEAR, month)
    if ds is None:
        continue

    era5_times = pd.to_datetime(ds['valid_time'].values)
    lats_e = ds['latitude'].values
    lons_e = ds['longitude'].values
    lv_dim = 'pressure_level' if 'pressure_level' in ds.dims else 'level'

    month_rows = gps_resampled[
        (gps_resampled['timestamp'].dt.month == month) &
        gps_resampled['lat'].notna()
    ].reset_index(drop=True)
    print(f'  [{YEAR}-{month:02d}] {len(month_rows):,}건 매칭 중...')

    month_records = []
    for _, row in month_rows.iterrows():
        lat_idx = int(np.argmin(np.abs(lats_e - row['lat'])))
        lon_idx = int(np.argmin(np.abs(lons_e - row['lon'])))
        tdiffs  = np.abs((era5_times - row['timestamp']).total_seconds())
        t_idx   = int(np.argmin(tdiffs))

        rec = {'bird': row['bird'], 'timestamp': row['timestamp']}
        if tdiffs[t_idx] <= 3600:
            for lv_i, lv in enumerate([1000, 925, 850]):
                for var in ['u','v','w','t','z','q','r','cc']:
                    try:
                        rec[f'{var}_{lv}'] = float(
                            ds[var].isel(
                                valid_time=t_idx,
                                latitude=lat_idx,
                                longitude=lon_idx,
                                **{lv_dim: lv_i}
                            ).values
                        )
                    except Exception:
                        rec[f'{var}_{lv}'] = np.nan
        month_records.append(rec)

    month_df = pd.DataFrame(month_records)
    month_df.to_csv(cache_path, index=False)
    print(f'  [{YEAR}-{month:02d}] 완료 → 중간 저장: {cache_path.name}')
    era5_records.append(month_df)

    ds.close()
    del ds
    gc.collect()

era5_df = pd.concat(era5_records, ignore_index=True)
print(f'\nERA5 매칭 완료: {len(era5_df):,}건')

ERA5 매칭 시작... (2015년)
  [2015-09] ERA5 로드 중...


/tmp/ipykernel_7079/2716049133.py:47: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 24. This could degrade performance. Instead, consider rechunking after loading.
  datasets.append(xr.open_dataset(wp, engine='netcdf4',
/tmp/ipykernel_7079/2716049133.py:47: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 24. This could degrade performance. Instead, consider rechunking after loading.
  datasets.append(xr.open_dataset(wp, engine='netcdf4',
/tmp/ipykernel_7079/2716049133.py:47: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 24. This could degrade performance. Instead, consider rechunking after loading.
  datasets.append(xr.open_dataset(wp, engine='netcdf4',
/tmp/ipykernel_7079/2716049133.py:47: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 24. This

  [2015-09] 4,941건 매칭 중...


In [ ]:
# ── Cell 8: GPS + ERA5 병합 및 파생 변수 계산 ─────────────────────────
print('병합 및 파생 변수 계산 중...')

df = gps_resampled.merge(era5_df, on=['bird','timestamp'], how='left')

# ERA5 보간 여부 표시
df['is_interpolated_era5'] = df['u_925'].isna().astype(int)

# ERA5 결측치 선형 보간 (개체별)
era5_cols = [f'{v}_{lv}' for lv in [1000,925,850]
             for v in ['u','v','w','t','z','q','r','cc']]
era5_cols = [c for c in era5_cols if c in df.columns]
for bird in df['bird'].unique():
    idx = df['bird'] == bird
    df.loc[idx, era5_cols] = (
        df.loc[idx, era5_cols]
        .interpolate(method='linear', limit_direction='both')
    )

# 파생 변수
for lv in [1000, 925, 850]:
    df[f'ws_{lv}']     = wind_support(df[f'u_{lv}'], df[f'v_{lv}'], df['heading'])
    df[f'wspeed_{lv}'] = np.sqrt(df[f'u_{lv}']**2 + df[f'v_{lv}']**2)
    df[f'wdir_{lv}']   = wind_dir(df[f'u_{lv}'], df[f'v_{lv}'])

df['lapse_rate'] = df['t_925'] - df['t_850']  # 거위 기준 (925-850)
df['is_moving']  = (df['ground_speed'].fillna(0) > 1).astype(int)

print(f'병합 완료: {len(df):,}행 × {len(df.columns)}컬럼')

In [ ]:
# ── Cell 9: 연도별 CSV 저장 ───────────────────────────────────────────
FINAL_COLS = [
    # 메타
    'bird', 'timestamp', 'is_interpolated_gps', 'is_interpolated_era5',
    # y 타깃
    'lat', 'lon', 'height_raw',
    # GPS 행동
    'ground_speed', 'heading', 'is_moving',
    # ERA5 원본 (8변수 × 3레벨 = 24개)
    'u_1000','v_1000','w_1000','t_1000','z_1000','q_1000','r_1000','cc_1000',
    'u_925', 'v_925', 'w_925', 't_925', 'z_925', 'q_925', 'r_925', 'cc_925',
    'u_850', 'v_850', 'w_850', 't_850', 'z_850', 'q_850', 'r_850', 'cc_850',
    # 파생 변수 (10개)
    'ws_1000','ws_925','ws_850',
    'wspeed_1000','wspeed_925','wspeed_850',
    'wdir_1000','wdir_925','wdir_850',
    'lapse_rate',
]
FINAL_COLS = [c for c in FINAL_COLS if c in df.columns]

df[FINAL_COLS].to_csv(out_csv, index=False)
mb = out_csv.stat().st_size / 1024**2
print(f'저장 완료: {out_csv.name}')
print(f'  행: {len(df):,} / 컬럼: {len(FINAL_COLS)} / 크기: {mb:.1f} MB')

In [ ]:
# ── Cell 10: 연도 처리 완료 확인 ─────────────────────────────────────
import pandas as pd

print(f'=== {YEAR}년 처리 완료 ===')
if out_csv.exists():
    df_check = pd.read_csv(out_csv)
    print(f'  파일  : {out_csv.name}')
    print(f'  행 수 : {len(df_check):,}')
    print(f'  컬럼 수: {len(df_check.columns)}')
    print(f'  개체 수: {df_check["bird"].nunique()}마리')
    print(f'  크기  : {out_csv.stat().st_size/1024**2:.1f} MB')
    print()
    print(f'  보간 비율 (GPS): {df_check["is_interpolated_gps"].mean()*100:.1f}%')
    print(f'  보간 비율 (ERA5): {df_check["is_interpolated_era5"].mean()*100:.1f}%')
    print(f'  lat NaN: {df_check["lat"].isna().sum()}건')

print()
print('=== 전체 진행 현황 ===')
for yr in [2014, 2015, 2016, 2017]:
    p = OUT_DIR / f'preprocessed_geese_{yr}.csv'
    status = f'완료 ({p.stat().st_size/1024**2:.1f} MB)' if p.exists() else '미완료'
    marker = '✅' if p.exists() else '⬜'
    print(f'  {marker} {yr}년: {status}')

if YEAR < 2017:
    print(f'\n다음 단계: Cell 4에서 YEAR = {YEAR+1} 로 변경 후 Cell 5~10 실행')
else:
    print('\n모든 연도 완료! Cell 11 실행하여 합치기')

In [ ]:
# ── Cell 11: 4개 연도 합치기 (모든 연도 완료 후 실행) ─────────────────
import pandas as pd

dfs = []
for year in [2014, 2015, 2016, 2017]:
    p = OUT_DIR / f'preprocessed_geese_{year}.csv'
    if not p.exists():
        print(f'[WARNING] {year}년 파일 없음 — 건너뜀')
        continue
    df_y = pd.read_csv(p)
    dfs.append(df_y)
    print(f'  {year}년: {len(df_y):,}행 로드')

full = pd.concat(dfs, ignore_index=True)
full = full.sort_values(['bird','timestamp']).reset_index(drop=True)

out_full = OUT_DIR / 'preprocessed_geese_full.csv'
full.to_csv(out_full, index=False)

print(f'\n최종 파일 저장 완료!')
print(f'  파일: {out_full.name}')
print(f'  총 {len(full):,}행 × {len(full.columns)}컬럼')
print(f'  크기: {out_full.stat().st_size/1024**2:.1f} MB')
print(f'  개체 수: {full["bird"].nunique()}마리')

# ── 샘플 파일 생성 (구조 확인용, Git 커밋용) ─────────────────────────
sample_path = OUT_DIR / 'preprocessed_geese_sample.csv'
full.head(100).to_csv(sample_path, index=False)
print(f'\n샘플 파일 생성: {sample_path.name} (100행)')

# ── 컬럼 목록 출력 ────────────────────────────────────────────────────
print(f'\n컬럼 목록:')
for i, col in enumerate(full.columns):
    print(f'  {i:2d}: {col}')

# ── 중간 캐시 파일 정리 (선택) ────────────────────────────────────────
cleanup = input('\n월별 캐시 파일 삭제하시겠어요? (y/n): ')
if cleanup.lower() == 'y':
    for f in OUT_DIR.glob('era5_matched_*.csv'):
        f.unlink()
        print(f'  삭제: {f.name}')
    print('캐시 파일 정리 완료!')